<a href="https://colab.research.google.com/github/swaran21/MiniProject/blob/main/pythonML/Recipe_Training_Improved_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍳 DistilGPT2 Recipe Generator Training

This notebook trains a DistilGPT2 model on the improved, consistently validated recipe dataset.

**Dataset:** `recipe_training_improved.txt` (~15,000 recipes)
**Model:** `distilgpt2`
**Goal:** Generate consistent recipes where titles match ingredients and instructions.

In [ ]:
# Install dependencies
!pip install transformers datasets torch

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 1. Upload the dataset
from google.colab import files
print("Upload recipe_training_improved.txt:")
uploaded = files.upload()

In [ ]:
# 2. Load Model and Tokenizer
model_name = "distilgpt2"
print(f"Loading {model_name}...")

tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# Add special tokens if needed (DistilGPT2 usually handles general text well, but we use <END>)
special_tokens = {'additional_special_tokens': ['<END>']}
tokenizer.add_special_tokens(special_tokens)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.to(device)
print("Model loaded.")

In [ ]:
# 3. Prepare Dataset
def load_dataset(file_path, tokenizer, block_size=400):
    return TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size,
        overwrite_cache=True,
    )

train_file = "recipe_training_improved.txt"

print("Processing dataset...")
train_dataset = load_dataset(train_file, tokenizer)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
)
print("Dataset ready.")

In [ ]:
# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./recipe_model",
    overwrite_output_dir=True,
    num_train_epochs=3,                # 3 epochs is usually enough for this size
    per_device_train_batch_size=8,     # Adjust based on GPU memory (8 or 16)
    save_steps=1000,
    save_total_limit=2,
    warmup_steps=200,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    prediction_loss_only=True,
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

In [ ]:
# 6. Train!
print("Starting training...")
trainer.train()
print("Training complete.")

In [ ]:
# 7. Save Model
output_path = "./recipe_gpt2_improved"
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)
print(f"Model saved to {output_path}")

In [ ]:
# 8. Test Generation
def generate_recipe(ingredients_input):
    prompt = f"INPUT: {ingredients_input}\nOUTPUT:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    outputs = model.generate(
        inputs.input_ids,
        max_length=400,
        num_return_sequences=1,
        temperature=0.7,
        top_p=0.85,
        repetition_penalty=1.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

# Test
print(generate_recipe("chicken, lemon, garlic"))
print("-"*50)
print(generate_recipe("tomatoes, mozzarella, basil"))

In [ ]:
# 9. Download Model
!zip -r recipe_gpt2_improved.zip ./recipe_gpt2_improved
files.download('recipe_gpt2_improved.zip')